# Feature-Based Unsupervised Anomaly Detection Walkthrough

This notebook uses the same single-record vibration dataset as `01_signal_analysis_walkthrough_v2.ipynb`, but keeps anomaly detection focused on interpretable sliding-window features.

The dataset is unlabeled, so every model output is an inspection ranking, not a confirmed fault label. Raw waveform chunk models, neural autoencoders, and spectrogram image classifiers are intentionally excluded because this record is short and produces only a few hundred overlapping windows.

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from signal_processing_prep.data_loading import SignalDatasetLoader
from signal_processing_prep.features import FeatureExtractor, FrequencyBand, SlidingWindowConfig
from signal_processing_prep.modeling import (
    DbscanOutlierScorer,
    IsolationForestScorer,
    LocalOutlierFactorScorer,
    OneClassSvmScorer,
    PcaReconstructionScorer,
    top_anomalies,
    RobustZScoreScorer,
    RobustMahalanobisScorer,
)
from signal_processing_prep.plotting import (
    plot_frequency_spectrum,
    plot_spectrogram_dynamic_range,
    plot_time_signal,
    plot_time_signal_adaptive,
    plot_wavelet_scalogram,
)
from signal_processing_prep.quality import assess_signal_quality
from signal_processing_prep.records import SignalRecord

plt.style.use("default")

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "vibration_anomaly_single_record.csv"
DATA_PATH

In [ ]:
from IPython import get_ipython

ip = get_ipython()
if ip is not None:
    try:
        ip.run_line_magic("matplotlib", "widget")
        print("Using matplotlib widget backend.")
    except Exception:
        ip.run_line_magic("matplotlib", "inline")
        print("Widget backend unavailable; using inline plots.")

## 2. Load The Same Dataset

In [ ]:
record = SignalDatasetLoader().load_file(DATA_PATH, signal_column="voltage")

print(record)
print(f"Samples: {record.n_samples:,}")
print(f"Duration: {record.duration_seconds:.2f} s")
print(f"Sampling rate: {record.sampling_rate_hz:.1f} Hz")
print(f"Label: {record.label}")

In [ ]:
quality = assess_signal_quality(record)
pd.DataFrame([quality.to_dict()])

## 3. Brief Signal Context

Before scoring windows, inspect the raw signal, global PSD, and spectrogram. These views explain what the feature-based detectors will later rank.

In [ ]:
fig, ax = plot_time_signal_adaptive(record, max_points=2000)
ax.set_ylabel("Voltage [V]");

In [ ]:
fig, ax = plot_frequency_spectrum(
    record,
    spectrum_type="psd",
    nperseg=4 * 4096,
)
ax.set_title("Full-record PSD overview")
ax.set_yscale("log");

In [ ]:
fig, ax = plot_spectrogram_dynamic_range(
    record,
    window_seconds=0.1,
    step_seconds=0.05,
    max_frequency_hz=6000,
    frequency_scale="log",
)
ax.set_title("Full-record spectrogram");

## 4. Shared Sliding-Window Feature Table

All detectors below use the same feature table. The windows match the v2 walkthrough: 0.20 seconds wide with a 0.025 second step. This gives hundreds of interpretable rows instead of high-dimensional raw waveform chunks.

In [ ]:
window_config = SlidingWindowConfig(
    window_seconds=0.20,
    step_seconds=0.025,
    frequency_bands=(
        FrequencyBand("rotating_40_500", 40.0, 500.0),
        FrequencyBand("resonance_1800_3200", 1800.0, 3200.0),
        FrequencyBand("broadband_3200_6000", 3200.0, 6000.0),
    ),
    frequency_window="hann",
    normalize_frequency_window_power=True,
)

feature_table = FeatureExtractor().extract_windows(record, window_config)
features = feature_table.to_dataframe()
print(f"Feature rows: {len(features):,}")
print(f"Feature columns: {len(features.columns):,}")
features.head()

In [ ]:
model_feature_columns = [
    column
    for column in features.select_dtypes(include=[np.number]).columns
    if column
    not in {
        "window_start_seconds",
        "window_end_seconds",
        "window_center_seconds",
        "window_n_samples",
        "sampling_rate_hz",
        "n_samples",
        "duration_seconds",
    }
]

model_feature_columns

## 5. Reference Hand-Built Score

This mirrors the v2 walkthrough: a transparent score from a few physically interpretable features. It is useful as a sanity check against generic detectors.

In [ ]:
reference_score_columns = [
    "rms",
    "crest_factor",
    "kurtosis",
    "band_energy_resonance_1800_3200",
    "band_energy_broadband_3200_6000",
]

reference = RobustZScoreScorer(feature_columns=tuple(reference_score_columns)).score(feature_table).prediction_frame
reference.sort_values("anomaly_score", ascending=False).head(10)

## 6. Generic Feature-Based Detectors

These methods return exploratory anomaly scores over windows. Robust Mahalanobis is used because candidate anomalies may contaminate ordinary mean and covariance estimates. It is fitted on a compact pair of non-degenerate features to avoid redundant near-deterministic feature relationships during covariance estimation.

In [ ]:
evaluations = {
    "reference_robust_z": reference,
    "robust_positive_z": RobustZScoreScorer().score(feature_table).prediction_frame,
    "robust_mahalanobis_distance": RobustMahalanobisScorer(
        feature_columns=('rms', "kurtosis", "crest_factor", "band_energy_broadband_3200_6000"),
        random_state=0,
    ).score(feature_table).prediction_frame,
    "isolation_forest": IsolationForestScorer(contamination=0.03, random_state=0).score(feature_table).prediction_frame,
    "one_class_svm": OneClassSvmScorer(nu=0.03).score(feature_table).prediction_frame,
    "local_outlier_factor": LocalOutlierFactorScorer(n_neighbors=10, contamination=0.03).score(feature_table).prediction_frame,
    "pca_reconstruction": PcaReconstructionScorer(n_components=0.95).score(feature_table).prediction_frame,
    # DBSCAN is eps-sensitive and included only as an exploratory clustering diagnostic.
    "dbscan_exploratory": DbscanOutlierScorer(eps=1.5, min_samples=8).score(feature_table).prediction_frame,
}

summary_rows = []
for method, predictions in evaluations.items():
    top = predictions.sort_values("anomaly_score", ascending=False).head(5)
    for _, row in top.iterrows():
        summary_rows.append(
            {
                "method": method,
                "rank": int(row.get("anomaly_rank", np.nan)),
                "window_start_seconds": row["window_start_seconds"],
                "window_end_seconds": row["window_end_seconds"],
                "window_center_seconds": row["window_center_seconds"],
                "anomaly_score": row["anomaly_score"],
            }
        )

top_by_method = pd.DataFrame(summary_rows)
top_by_method

In [ ]:
#evaluations["isolation_forest"].sort_values("anomaly_score", ascending=False).head(10)
plt.figure(figsize=(10, 4))
plt.plot(evaluations["reference_robust_z"]["window_center_seconds"], evaluations["reference_robust_z"]["anomaly_score"]*200)
plt.plot(evaluations["robust_mahalanobis_distance"]["window_center_seconds"], evaluations["robust_mahalanobis_distance"]["anomaly_score"])

In [ ]:
def minmax_score_for_display(values: pd.Series) -> pd.Series:
    values = pd.to_numeric(values, errors="coerce")
    low = values.min()
    high = values.max()
    if not np.isfinite(low) or not np.isfinite(high) or high == low:
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - low) / (high - low)

score_frame = features[["window_start_seconds", "window_end_seconds", "window_center_seconds"]].copy()
score_frame["row_index"] = features.index
rank_frame = score_frame.copy()

for method, predictions in evaluations.items():
    aligned = predictions[["row_index", "anomaly_score"]].copy()
    aligned[f"{method}_score"] = minmax_score_for_display(aligned["anomaly_score"])
    aligned[f"{method}_rank"] = aligned["anomaly_score"].rank(method="first", ascending=False)
    score_frame = score_frame.merge(
        aligned[["row_index", f"{method}_score"]],
        on="row_index",
        how="left",
    ).rename(columns={f"{method}_score": method})
    rank_frame = rank_frame.merge(
        aligned[["row_index", f"{method}_rank"]],
        on="row_index",
        how="left",
    ).rename(columns={f"{method}_rank": method})

score_columns = list(evaluations)
score_frame["consensus_score"] = score_frame[score_columns].mean(axis=1)
score_frame["consensus_rank"] = score_frame["consensus_score"].rank(method="first", ascending=False).astype(int)

consensus = score_frame.sort_values("consensus_score", ascending=False)
consensus.head(15)

## 7. Score Timelines And Agreement

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for method in score_columns:
    ax.plot(
        score_frame["window_center_seconds"],
        score_frame[method],
        linewidth=1.0,
        alpha=0.8,
        label=method,
    )
ax.plot(
    score_frame["window_center_seconds"],
    score_frame["consensus_score"],
    color="black",
    linewidth=2.0,
    label="consensus_score",
)
ax.set_title("Normalized anomaly scores by window")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Min-max normalized score")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right", ncols=2)
fig.tight_layout();

In [ ]:
agreement = score_frame[["window_start_seconds", "window_end_seconds", "window_center_seconds", "consensus_score", "consensus_rank"]].copy()
for method in score_columns:
    agreement[f"{method}_top_20"] = rank_frame[method] <= 20
agreement["top_20_votes"] = agreement[[f"{method}_top_20" for method in score_columns]].sum(axis=1)

agreement.sort_values(["top_20_votes", "consensus_score"], ascending=False).head(20)

## 8. Inspect The Top Consensus Region

The consensus window is still only a candidate. Inspect the raw waveform and localized time-frequency views before making any engineering claim.

In [ ]:
top_consensus = consensus.iloc[0]
candidate_center = float(top_consensus["window_center_seconds"])
zoom_start = max(candidate_center - 0.6, 0.0)
zoom_duration = 1.2

print(
    "Top consensus candidate: "
    f"{top_consensus['window_start_seconds']:.3f} s to "
    f"{top_consensus['window_end_seconds']:.3f} s, "
    f"center {candidate_center:.3f} s"
)

fig, ax = plot_time_signal(
    record,
    start_seconds=zoom_start,
    duration_seconds=zoom_duration,
    max_points=8000,
)
ax.axvspan(
    float(top_consensus["window_start_seconds"]),
    float(top_consensus["window_end_seconds"]),
    color="tab:red",
    alpha=0.2,
    label="top consensus window",
)
ax.set_ylabel("Voltage [V]")
ax.set_title("Raw signal around top consensus candidate")
ax.legend(loc="best");

In [ ]:
start_index = int(round(zoom_start * record.sampling_rate_hz))
end_index = int(round((zoom_start + zoom_duration) * record.sampling_rate_hz))
end_index = min(end_index, record.n_samples)

zoom_record = record.segment(start_index, end_index, index=0)

fig, ax = plot_spectrogram_dynamic_range(
    zoom_record,
    window_seconds=0.05,
    step_seconds=0.0025,
    max_frequency_hz=6000,
    frequency_scale="log",
)
ax.set_title("Candidate-region spectrogram; time is relative to zoom window");

In [ ]:
fig, ax = plot_wavelet_scalogram(
    zoom_record,
    min_frequency_hz=20.0,
    max_frequency_hz=6000.0,
    n_frequencies=128,
    frequency_scale="log",
)
ax.set_title("Candidate-region Morlet wavelet scalogram; time is relative to zoom window");

## 9. Interpretation Limits

This notebook compares feature-based unsupervised rankings on one unlabeled record. The strongest result is agreement about which windows deserve inspection.

Important limitations:

- There is only one record and no confirmed anomaly labels.
- Sliding windows overlap, so rows are not independent samples.
- Detector parameters such as contamination, `nu`, neighbor count, PCA variance, and DBSCAN `eps` affect rankings.
- DBSCAN is included as an exploratory clustering diagnostic only.
- These scores identify candidate windows, not faults. A real validation step would need labeled events, repeated acquisitions, or independent operating-condition metadata.